# CardioIA – Fase 2
## Parte 1 – Extração de Sintomas e Sugestão de Diagnóstico

Este notebook irá funcionar localmente e no Google Colab, localizando automaticamente a pasta `data`.

In [1]:
!git clone https://github.com/limadeivisson/cardioia-fase2-diagnostico-automatizado.git

Cloning into 'cardioia-fase2-diagnostico-automatizado'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 21 (delta 3), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 128.97 KiB | 32.24 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [2]:
import pandas as pd
from pathlib import Path

In [3]:
def localizar_base():
    candidatos = [
        Path('.'),
        Path('cardioia-fase2-diagnostico-automatizado'),
        Path('/content/cardioia-fase2-diagnostico-automatizado'),
    ]
    for candidato in candidatos:
        if (candidato / 'data').exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível localizar a pasta 'data'. Se estiver no Colab, clone o repositório primeiro."
    )

base = localizar_base()
print('Base encontrada em:', base.resolve())

Base encontrada em: /content/cardioia-fase2-diagnostico-automatizado


In [4]:
frases_path = base / 'data' / 'sintomas_pacientes.txt'
mapa_path = base / 'data' / 'mapa_conhecimento_sintomas_doencas.csv'

with open(frases_path, 'r', encoding='utf-8') as f:
    frases = [linha.strip() for linha in f.readlines() if linha.strip()]

mapa = pd.read_csv(mapa_path)
print('Quantidade de frases:', len(frases))
display(mapa.head())

Quantidade de frases: 10


,sintoma_1,sintoma_2,doenca_associada
0,dor no peito,aperto no tórax,Infarto
1,cansaço constante,fadiga,Insuficiência Cardíaca
2,falta de ar,dificuldade para respirar,Angina
3,palpitações,coração acelerado,Arritmia
4,inchaço nas pernas,edema,Insuficiência Cardíaca


In [5]:
def identificar_sintomas(frase, mapa):
    frase_lower = frase.lower()
    sintomas = []
    doencas = []

    for _, row in mapa.iterrows():
        s1 = str(row['sintoma_1']).lower()
        s2 = str(row['sintoma_2']).lower()
        doenca = row['doenca_associada']

        if s1 in frase_lower:
            sintomas.append(row['sintoma_1'])
            doencas.append(doenca)
        if s2 in frase_lower:
            sintomas.append(row['sintoma_2'])
            doencas.append(doenca)

    diagnostico = pd.Series(doencas).value_counts().idxmax() if doencas else 'Sem diagnóstico'
    return sintomas, diagnostico

In [6]:
resultados = []
for frase in frases:
    sintomas, diagnostico = identificar_sintomas(frase, mapa)
    resultados.append({
        'frase': frase,
        'sintomas_encontrados': ', '.join(sintomas) if sintomas else 'Nenhum',
        'diagnostico_sugerido': diagnostico
    })

resultado_df = pd.DataFrame(resultados)
display(resultado_df)

,frase,sintomas_encontrados,diagnostico_sugerido
0,Há dois dias estou com uma dor no peito que pi...,dor no peito,Infarto
1,Sinto cansaço constante há uma semana mesmo ap...,cansaço constante,Insuficiência Cardíaca
2,Tenho falta de ar desde ontem quando caminho p...,falta de ar,Angina
3,Estou com palpitações frequentes e sensação de...,"palpitações, coração acelerado",Arritmia
4,Notei um aperto no tórax ao subir escadas hoje...,aperto no tórax,Infarto
5,Tenho inchaço nas pernas e cansaço intenso nos...,inchaço nas pernas,Insuficiência Cardíaca
6,Sinto tontura e quase desmaio ao me levantar r...,"tontura, desmaio",Arritmia
7,Estou com dor no braço esquerdo junto com pres...,"dor no braço esquerdo, pressão no peito",Infarto
8,Tenho dificuldade para respirar e chiado no pe...,"dificuldade para respirar, chiado no peito",Angina
9,Percebo fadiga intensa e mal-estar persistente...,"fadiga, mal-estar",Insuficiência Cardíaca


### Se estiver no Google Colab
Execute antes:

```python
!git clone https://github.com/limadeivisson/cardioia-fase2-diagnostico-automatizado.git
```